In [ ]:
import dgl.function as fn
import networkx as nx
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import category_encoders as ce
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# !!! Important: run_training overwrites best_edge_sage.pt model
meaning: if we train best model, this one will overwrite it. So make sure to preserve it. or re-run after the complexity test

In [ ]:
from data_cleaning import clean_nfunsw_nb15

# load this dataset and clean it
data = pd.read_csv('data/NF-UNSW-NB15-v3.csv')

data = clean_nfunsw_nb15(data)

[clean] Step 1/3: dropped rows missing IP/ports: 0 (from 2365424 -> 2365424)
[clean] Step 2/3: columns with ±inf and/or missing values detected:
        ['DST_TO_SRC_SECOND_BYTES', 'SRC_TO_DST_SECOND_BYTES']
[clean] Step 3/3: filled 244986 missing numeric values with 0.
[clean] Done. Final shape: (2365424, 55)


In [5]:
data.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1,45870,149.171.126.1,53,17,5.0,130,2,...,0,0,0,0,0,0,0,0,0,Benign


In [ ]:
# anonymize IP addresses by replacing them with random private IPs in the range
# To construct the network graph from the flow data, we mapped the original source IP addresses to randomly assigned
# IP addresses in the range from 172.16.0.1 to 172.31.0.1. The reason for this is the fact that in a lot of the NIDS datasets
# only a small number of IP addresses were used as the source of the attacks. The random mapping avoids
# the potential problem of the source IP addresses providing an unintentional label for attack traffic.
# NOTE: how to modify original randomizer code to temporally consistent flows (i.e., single src to dst should have matching dst to src flow)
#data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001))))

In [7]:
# convert IP addresses and ports to string type then concatenate them to form unique node identifiers
data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(str)
data['L4_SRC_PORT'] = data.L4_SRC_PORT.apply(str)
data['IPV4_DST_ADDR'] = data.IPV4_DST_ADDR.apply(str)
data['L4_DST_PORT'] = data.L4_DST_PORT.apply(str)

data['IPV4_SRC_ADDR'] = data['IPV4_SRC_ADDR'] + ':' + data['L4_SRC_PORT']
data['IPV4_DST_ADDR'] = data['IPV4_DST_ADDR'] + ':' + data['L4_DST_PORT']

data.drop(columns=['L4_SRC_PORT','L4_DST_PORT'],inplace=True)
data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2365424 entries, 0 to 2365423
Data columns (total 53 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   FLOW_START_MILLISECONDS      int64  
 1   FLOW_END_MILLISECONDS        int64  
 2   IPV4_SRC_ADDR                object 
 3   IPV4_DST_ADDR                object 
 4   PROTOCOL                     int64  
 5   L7_PROTO                     float64
 6   IN_BYTES                     int64  
 7   IN_PKTS                      int64  
 8   OUT_BYTES                    int64  
 9   OUT_PKTS                     int64  
 10  TCP_FLAGS                    int64  
 11  CLIENT_TCP_FLAGS             int64  
 12  SERVER_TCP_FLAGS             int64  
 13  FLOW_DURATION_MILLISECONDS   int64  
 14  DURATION_IN                  int64  
 15  DURATION_OUT                 int64  
 16  MIN_TTL                      int64  
 17  MAX_TTL                      int64  
 18  LONGEST_FLOW_PKT             int64  
 19  SHORT

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,IPV4_DST_ADDR,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2:4894,149.171.126.3:53,17,5.0,146,2,178,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4:52671,149.171.126.6:31992,6,11.0,4704,28,2976,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0:47290,149.171.126.9:6881,6,37.0,13662,238,548216,438,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8:43310,149.171.126.7:53,17,5.0,146,2,178,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1:45870,149.171.126.1:53,17,5.0,130,2,162,2,...,0,0,0,0,0,0,0,0,0,Benign


In [8]:
# we don't need the Label column as we are doing multiclass classification
data.drop(columns=['Label'],inplace = True)

In [9]:
data.rename(columns={"Attack": "label"},inplace = True)

## Chronological load data to 60/30/10

In [10]:
from chronological_split import load_split_indices

# 3) In any future run (graph build, feature store, training), load the SAME indices:
train_idx2, val_idx2, test_idx2, meta2 = load_split_indices("artifacts/splits")

# Use them to subset clean dataframe (data) deterministically
df_train = data.loc[train_idx2]
df_val   = data.loc[val_idx2]
df_test  = data.loc[test_idx2]

[split] Chronological 60/30/10 with boundary safety
        TRAIN:  1419254  t∈[1421927376907, 1424229622767]
        VAL:     709628  t∈(1424229622767, 1424254516414]
        TEST:    236542  t∈(1424254516414, 1424262564927]
[split] Saved indices → artifacts/splits/split_indices.npz and meta → artifacts/splits/meta.json
[split] Saved materialized splits to artifacts/splits (parquet).


## Label encoding

In [11]:
from label_mapping import fit_label_map, transform_labels, save_label_map, load_label_map, class_weights_from_train

# 1) Fit mapping on TRAIN ONLY
label2id = fit_label_map(df_train["label"], order="alpha")  # or order="freq"
save_label_map("artifacts/label_map.json", label2id)

# 2) Apply mapping to all splits (consistent)
label2id = load_label_map("artifacts/label_map.json")
y_train = transform_labels(df_train["label"], label2id)  # np.int64
y_val   = transform_labels(df_val["label"], label2id)
y_test  = transform_labels(df_test["label"], label2id)

# (Optional) sanity check: ensure no -1 slipped in
assert (y_train >= 0).all(), "Train split has unseen/missing labels."

# 3) Class weights for loss
weights = class_weights_from_train(y_train, num_classes=len(label2id))

## Numerical transformation
### Remove highly corelated features
This function visualise correlation between given features.
Also performs log1p, scaler, zero-variance drop, optional Spearman prune on numericals
Applies transformation to test and validation

In [ ]:
from corr_visualisation import plot_numeric_corr_heatmap
from feature_numeric import fit_numeric_transform, transform_numeric

# Toggle correlation pruning
APPLY_CORR_PRUNE = True   # set to False to disable pruning and re-run preprocessing
# if you want to check sensitivity of final results to pruning make sure to run train model on best parameters only. and report results.

# 0) Columns to EXCLUDE here because they’re numeric-coded categoricals:
numeric_cats = [
    "PROTOCOL", "L7_PROTO",
    "ICMP_TYPE", "ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE", "DNS_QUERY_ID",
    "FTP_COMMAND_RET_CODE",
    # any other *_TYPE / *_ID style columns you want one-hot later
]

# 1) Correlation heatmap (pre-pruning) on TRAIN only
viz_info = plot_numeric_corr_heatmap(
    df_train,
    exclude_numeric_categoricals=numeric_cats,
    out_dir="artifacts/corr",
    threshold=0.995,         # same ballpark as your pruning threshold
    max_features=150,        # avoid unreadable giant plots
    nonneg_frac=0.995,       # infer nonnegative cols for log1p
    topk_pairs=100,
    filename_prefix="spearman_corr_train"
)
print("[corr-viz]", viz_info)

# 2) TRAIN: fit numeric pipeline (log1p, scaler, zero-variance drop, optional Spearman prune)
Xnum_train, num_arts = fit_numeric_transform(
    df_train,
    exclude_numeric_categoricals=numeric_cats,
    scaler_type="standard",           # or "robust" if outliers are extreme
    apply_corr_prune=APPLY_CORR_PRUNE,
    corr_threshold=0.995,
    artifacts_dir="artifacts/numeric"
)
print(
    "[numeric] dimensions:",
    "before=", num_arts.dim_before_var,
    "after_variance=", num_arts.dim_after_var,
    "after_correlation=", num_arts.dim_after_corr,
    "Pruning=", APPLY_CORR_PRUNE,
)
print("[numeric] train shape:", Xnum_train.shape)

# 2) VAL/TEST: apply frozen transforms
Xnum_val, _  = transform_numeric(df_val,  artifacts_dir="artifacts/numeric")
Xnum_test, _ = transform_numeric(df_test, artifacts_dir="artifacts/numeric")
print("[numeric] val/test shapes:", Xnum_val.shape, Xnum_test.shape)

[corr-viz] {'n_numeric': 42, 'n_plotted': 42, 'threshold': 0.995, 'nonneg_frac': 0.995, 'png_path': 'artifacts/corr\\spearman_corr_train_full.png', 'pairs_csv': 'artifacts/corr\\spearman_corr_train_top_pairs.csv'}
[numeric] train shape: (1419254, 38)
[numeric] val/test shapes: (709628, 38) (236542, 38)


## Categorical encoding
fits a OneHotEncoder on TRAIN only (sparse CSR, float32),
handles numeric-coded categoricals (e.g., PROTOCOL, L7_PROTO, *_TYPE, *_ID),
Optionally puts IPv4 ports in buckets, collapses rare categories (by min_freq or top_k) - not used here as we droped IPv4 ports after encoding it to unique hosts with IP addresses

persists artifacts (encoder + per-column metadata) for deterministic transforms,
transforms VAL/TEST,
returns CSR matrices ready to stack with numeric transformation.

In [13]:
from categorical_encoding import fit_categorical_transform, transform_categorical

# Choose categorical columns (include your numeric-coded categoricals here!)
cat_cols = [
    "PROTOCOL", "L7_PROTO",
    "ICMP_TYPE", "ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE", "DNS_QUERY_ID",
    "FTP_COMMAND_RET_CODE",
    # If you want ports as categories (after port bucketing):
    # "L4_SRC_PORT", "L4_DST_PORT", # we dropped ports after concatenating them to IP addresses
    # any other *_TYPE / *_ID style columns you want one-hot
]

# 1) TRAIN — fit encoder (optionally collapse rare cats)
Xcat_train, cat_arts = fit_categorical_transform(
    df_train,
    cat_cols=cat_cols,
    use_port_buckets=False,     # buckets ports into IANA groups to avoid huge dims
    min_freq=50,               # collapse categories with <50 occurrences to "__RARE__" (tune as needed)
    top_k=None,                # or e.g., top_k=20 to keep top-20 per column
    artifacts_dir="artifacts/categorical",
)
print("[categorical] train CSR shape:", Xcat_train.shape)

# 2) VAL/TEST — apply frozen encoder
Xcat_val  = transform_categorical(df_val,  artifacts_dir="artifacts/categorical")
Xcat_test = transform_categorical(df_test, artifacts_dir="artifacts/categorical")
print("[categorical] val/test CSR shapes:", Xcat_val.shape, Xcat_test.shape)


[categorical] train CSR shape: (1419254, 563)
[categorical] val/test CSR shapes: (709628, 563) (236542, 563)


## Prepare feature store for mini-batch and other
What: Persist transformed features by split, not in the graph.
Why: Memory safety; fast random access by batch.

a) mini-batch with neighbrour sampling (k-layers)
b) random walk

In [ ]:
from feature_store import build_feature_store

# We have:
# df_train, df_val, df_test       (chronological splits)
# y_train, y_val, y_test          (int labels via label_mapping)
# train_idx, val_idx, test_idx    (global edge indices from persist_splits)

numeric_cats = [
    "PROTOCOL","L7_PROTO","ICMP_TYPE","ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE","DNS_QUERY_ID","FTP_COMMAND_RET_CODE",
    #"L4_SRC_PORT","L4_DST_PORT",  # if you bucket/one-hot ports - otherwise remove
]

cat_cols = numeric_cats  # plus any extra true categoricals if you have them

shapes = build_feature_store(
    df_train, df_val, df_test,
    y_train, y_val, y_test,
    train_idx2, val_idx2, test_idx2,
    numeric_categoricals=numeric_cats,
    categorical_cols=cat_cols,
    out_dir="feature_store",
    numeric_artifacts_dir="artifacts/numeric",
    categorical_artifacts_dir="artifacts/categorical",
    use_port_buckets=False,
    rare_min_freq=50,
    rare_top_k=None,
    save_timestamps=True,
)
print(shapes)

[feature_store] train: n=1419254 d_num=38 d_cat=563
[feature_store] val  : n=709628   d_num=38   d_cat=563
[feature_store] test : n=236542  d_num=38  d_cat=563
{'train': (1419254, 38, 563), 'val': (709628, 38, 563), 'test': (236542, 38, 563)}


## Graph construction
What: Build DGL graph without edge features.
Why: Keep memory low; features come from the store per batch (fixed k-hop) or Random walk

No reverse duplication: we add exactly one directed edge per flow from the dataset; reverse flows (if present) appear naturally as their own edges later in time.

Alignment: g.edata[dgl.EID] stores the global edge IDs (original row indices) so your dataloader can fetch edge features from the feature store by ID in every batch.

Minimal memory: edge features live off-graph in the feature store; the graph only carries labels and timestamps.

In [ ]:
import copy, os, json
from types import SimpleNamespace
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from graph_build import build_light_graph_for_split
from feature_store import build_feature_store
from train_edgecls_dbg import run_training
import torch

FRACS = [0.25, 0.5, 0.75]

for FRAC in FRACS:
    print(f"\n================ FRAC = {FRAC:.2f} ================\n")

    # ---------- 1) SMALL TRAIN (stratified) ----------
    train_pos = np.arange(len(df_train))
    sss_tr = StratifiedShuffleSplit(
        n_splits=1,
        train_size=FRAC,
        random_state=42,
    )
    small_train_pos, _ = next(sss_tr.split(train_pos, y_train))
    small_train_pos = np.sort(small_train_pos)

    train_idx2_arr = train_idx2.to_numpy() if hasattr(train_idx2, "to_numpy") else np.asarray(train_idx2)
    train_sampled = np.sort(train_idx2_arr[small_train_pos])
    train_idx_small = pd.Index(train_sampled)

    print(
        f"[FRAC={FRAC:.2f}] Using {len(train_idx_small)} train edges out of {len(train_idx2)} "
        f"(~{FRAC*100:.1f}%) with stratified sampling"
    )

    df_train_small = data.loc[train_idx_small]
    y_train_small  = y_train[small_train_pos]

    # ---------- 2) SMALL VAL (stratified) ----------
    val_pos = np.arange(len(df_val))
    sss_va = StratifiedShuffleSplit(
        n_splits=1,
        train_size=FRAC,
        random_state=43,
    )
    small_val_pos, _ = next(sss_va.split(val_pos, y_val))
    small_val_pos = np.sort(small_val_pos)

    val_idx2_arr = val_idx2.to_numpy() if hasattr(val_idx2, "to_numpy") else np.asarray(val_idx2)
    val_sampled = np.sort(val_idx2_arr[small_val_pos])
    val_idx_small = pd.Index(val_sampled)

    print(
        f"[FRAC={FRAC:.2f}] Using {len(val_idx_small)} val edges out of {len(val_idx2)} "
        f"(~{FRAC*100:.1f}%) with stratified sampling"
    )

    df_val_small = data.loc[val_idx_small]
    y_val_small  = y_val[small_val_pos]

    # ---------- 3) Small feature store (train+val small, test full) ----------
    small_fs_dir = f"complexity_fs_frac{int(FRAC*100)}"
    os.makedirs(small_fs_dir, exist_ok=True)

    numeric_cats = [
        "PROTOCOL","L7_PROTO","ICMP_TYPE","ICMP_IPV4_TYPE",
        "DNS_QUERY_TYPE","DNS_QUERY_ID","FTP_COMMAND_RET_CODE",
    ]
    cat_cols = numeric_cats

    shapes_small = build_feature_store(
        df_train_small, df_val_small, df_test,
        y_train_small, y_val_small, y_test,
        train_idx_small, val_idx_small, test_idx2,
        numeric_categoricals=numeric_cats,
        categorical_cols=cat_cols,
        out_dir=small_fs_dir,
        numeric_artifacts_dir="artifacts/numeric",
        categorical_artifacts_dir="artifacts/categorical",
        use_port_buckets=False,
        rare_min_freq=50,
        rare_top_k=None,
        save_timestamps=True,
    )
    print(f"[FRAC={FRAC:.2f}] [small fs] shapes:", shapes_small)

    # ---------- 4) Small graphs ----------
    graphs_dir = f"graphs_small_frac{int(FRAC*100)}"
    os.makedirs(graphs_dir, exist_ok=True)

    g_train_small, ip2id_small = build_light_graph_for_split(
        df_train_small, os.path.join(small_fs_dir, "train"),
        ip2id=None, device="cpu",
        save_path=os.path.join(graphs_dir, "train.bin"),
    )
    g_val_small, ip2id_small = build_light_graph_for_split(
        df_val_small, os.path.join(small_fs_dir, "val"),
        ip2id=ip2id_small, device="cpu",
        save_path=os.path.join(graphs_dir, "val.bin"),
    )

    print(
        f"[FRAC={FRAC:.2f}] Small train graph: nodes = {g_train_small.num_nodes()}, edges = {g_train_small.num_edges()}"
    )
    print(
        f"[FRAC={FRAC:.2f}] Small val graph:   nodes = {g_val_small.num_nodes()}, edges = {g_val_small.num_edges()}"
    )

    # ---------- 5) Training for this FRAC ----------
    base_args = SimpleNamespace(
        feature_store=small_fs_dir,
        graphs_dir=graphs_dir,
        split_train="train",
        split_val="val",
        hidden=128,
        layers=2,
        aggregator="mean",
        edge_in=0,
        edge_mlp_hidden=128,
        dropout=0.4,
        fanouts="15,10",
        batch_size=256,
        epochs=25,
        lr=3e-4,
        weight_decay=1e-4,
        device="cuda" if torch.cuda.is_available() else "cpu",
        num_workers=0,
        seed=42,
        debug=False,
    )

    run_name = f"compl_frac{int(FRAC*100)}_hid{base_args.hidden}_agg{base_args.aggregator}_fan15-10_drop40_bs256"
    print(f"[FRAC={FRAC:.2f}] === RUN: {run_name} ===")

    res = run_training(copy.deepcopy(base_args))
    best_acc = float(res.get("best_val_acc", -1.0))
    history = res.get("history", None)

    os.makedirs("artifacts/compl_logs", exist_ok=True)
    results_path = f"artifacts/compl_logs/{run_name}_history.json"
    if history is not None:
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)
        print(f"[FRAC={FRAC:.2f}] Saved history -> {results_path}")
    print(f"[FRAC={FRAC:.2f}] best_val_acc = {best_acc:.4f}")

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt

# same FRACS as used in training
FRACS = [0.25, 0.5, 0.75]

def load_epoch_times(history_path):
    """Return total train_time and total val_time from a history JSON."""
    with open(history_path, "r", encoding="utf-8") as f:
        hist = json.load(f)

    # hist is expected to be a list[dict] with keys "train_time", "val_time"
    train_times = []
    val_times = []

    if isinstance(hist, list):
        for ep in hist:
            if isinstance(ep, dict):
                train_times.append(ep.get("train_time", 0.0))
                val_times.append(ep.get("val_time", 0.0))
    elif isinstance(hist, dict):
        # fallback to old dict-style format, if any
        train_times = hist.get("train_time", [])
        val_times = hist.get("val_time", [])

    # sum over epochs
    total_train = float(np.sum(train_times)) if len(train_times) > 0 else 0.0
    total_val = float(np.sum(val_times)) if len(val_times) > 0 else 0.0
    return total_train, total_val

# 1) Collect times for each fractional run
frac_vals = []
train_times = []
val_times = []

for FRAC in FRACS:
    run_name = f"compl_frac{int(FRAC*100)}_hid128_aggmean_fan15-10_drop40_bs256"
    history_path = os.path.join("artifacts", "compl_logs", f"{run_name}_history.json")

    if not os.path.exists(history_path):
        print(f"[WARN] Missing history for FRAC={FRAC:.2f}: {history_path}")
        continue

    t_train, t_val = load_epoch_times(history_path)
    frac_vals.append(FRAC)
    train_times.append(t_train)
    val_times.append(t_val)
    print(f"[FRAC={FRAC:.2f}] total train_time={t_train:.1f}s, val_time={t_val:.1f}s")

# # 2) Add the full-graph point (4th point)
# # adjust this path and FRAC value to how you saved the full run
# FULL_FRAC = 1.0
# FULL_HISTORY = "artifacts/best_history.json"  # <-- change if needed

# full_t_train, full_t_val = load_epoch_times(FULL_HISTORY)
# frac_vals.append(FULL_FRAC)
# train_times.append(full_t_train)
# val_times.append(full_t_val)
# print(f"[FULL] total train_time={full_t_train:.1f}s, val_time={full_t_val:.1f}s")

# 3) Sort by FRAC so lines are monotonic in x
order = np.argsort(frac_vals)
frac_vals = np.array(frac_vals)[order]
train_times = np.array(train_times)[order]
val_times = np.array(val_times)[order]

# 4) Plot: x = time, y = FRAC
plt.figure(figsize=(8, 6))

# training time line
plt.plot(
    train_times,
    frac_vals,
    marker="o",
    linestyle="-",
    color="tab:blue",
    label="Training time",
)

# validation time line
plt.plot(
    val_times,
    frac_vals,
    marker="s",
    linestyle="--",
    color="tab:orange",
    label="Validation time",
)

# # highlight the full run point (last x,y)
# plt.scatter(
#     train_times[-1],
#     frac_vals[-1],
#     color="tab:blue",
#     edgecolor="black",
#     s=80,
#     zorder=5,
# )
# plt.scatter(
#     val_times[-1],
#     frac_vals[-1],
#     color="tab:orange",
#     edgecolor="black",
#     s=80,
#     zorder=5,
# )

plt.xlabel("Total time [s]")
plt.ylabel("% of Nodes in graph")
plt.title("Complexity: training/validation time vs. used data fraction")
plt.grid(True, linestyle="--", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()